# Calibration: classical vs surrogate

Both methods recover Heston parameters from the same synthetic surfaces. Classical runs Nelder-Mead on the COS pricer; the surrogate does gradient descent on the frozen network's inputs.

In [ ]:
import os, sys
os.chdir("..")
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.model_calibration import run_comparison, summarise, synthetic_target, surrogate_calibrate
from src.heston import HESTON_PARAM_NAMES, PARAM_RANGES
from src.data_gen import surface_for_params, MONEYNESS_GRID, MATURITY_GRID
from src.train_surrogate import load_model

model, y_scaler = load_model()

## Run the comparison

Ten trials, new ground-truth parameters each time, both starting from the midpoint of the ranges.

In [ ]:
rows = run_comparison(n_trials=10)
summary = summarise(rows)
summary

## Time and accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

trials = [r["trial"] for r in rows]
axes[0].bar([t - 0.2 for t in trials], [r["classical_time"] for r in rows], 0.4, label="classical")
axes[0].bar([t + 0.2 for t in trials], [r["surrogate_time"] for r in rows], 0.4, label="surrogate")
axes[0].set_yscale("log")
axes[0].set_xlabel("trial"); axes[0].set_ylabel("seconds (log)")
axes[0].set_title(f"{summary['speedup']:.0f}x faster")
axes[0].legend()

axes[1].bar([t - 0.2 for t in trials], [r["classical_err"] for r in rows], 0.4, label="classical")
axes[1].bar([t + 0.2 for t in trials], [r["surrogate_err"] for r in rows], 0.4, label="surrogate")
axes[1].set_xlabel("trial"); axes[1].set_ylabel("mean param error (% of range)")
axes[1].set_title("parameter recovery")
axes[1].legend()

plt.tight_layout()

## Which parameters are hard to recover

Error per parameter, averaged over trials. kappa is the usual offender: the surface is fairly insensitive to it, so very different values fit almost equally well.

In [ ]:
widths = np.array([hi - lo for lo, hi in PARAM_RANGES.values()])
classical = np.array([np.abs(r["classical_params"] - r["true_params"]) / widths for r in rows])
surrogate = np.array([np.abs(r["surrogate_params"] - r["true_params"]) / widths for r in rows])

per_param = pd.DataFrame({"classical": 100 * classical.mean(axis=0),
                          "surrogate": 100 * surrogate.mean(axis=0)},
                         index=HESTON_PARAM_NAMES)

per_param.plot.bar(figsize=(7, 4), rot=0)
plt.ylabel("mean error (% of range)"); plt.title("error by parameter")
per_param.round(2)

## Does the fitted surface actually match?

Parameter error is not the whole story. A different parameter set can still reproduce the surface, which is what calibration is really asked to do.

In [ ]:
r = rows[0]
true_surface = surface_for_params(*r["true_params"])
classical_surface = surface_for_params(*r["classical_params"])
surrogate_surface = surface_for_params(*r["surrogate_params"])

shape = (len(MONEYNESS_GRID), len(MATURITY_GRID))
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, surf, name in zip(axes,
                          [true_surface, classical_surface, surrogate_surface],
                          ["target", "classical fit", "surrogate fit"]):
    grid = surf.reshape(shape)
    for j, T in enumerate(MATURITY_GRID):
        ax.plot(MONEYNESS_GRID, grid[:, j], marker=".", label=f"T={T}")
    ax.set_title(name); ax.set_xlabel("moneyness")
axes[0].set_ylabel("implied vol"); axes[0].legend(fontsize=7)
plt.tight_layout()

for name, surf in [("classical", classical_surface), ("surrogate", surrogate_surface)]:
    rmse = np.sqrt(((surf - true_surface) ** 2).mean())
    print(f"{name:10s} surface RMSE {rmse:.5f} vol ({rmse*100:.3f} vol points)")

## Convergence

How far the surrogate gets as a function of gradient steps, and what that buys in wall-clock.

In [ ]:
true_params, target = synthetic_target(999)
truth = np.array([true_params[n] for n in HESTON_PARAM_NAMES])

history = []
for steps in [10, 25, 50, 100, 200, 400, 800]:
    params, elapsed, _ = surrogate_calibrate(target, model, y_scaler, steps=steps)
    history.append({"steps": steps, "seconds": elapsed,
                    "err_pct": 100 * (np.abs(params - truth) / widths).mean()})

history = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["steps"], history["err_pct"], marker="o")
ax.set_xscale("log"); ax.set_xlabel("gradient steps"); ax.set_ylabel("param error (% of range)")
ax.grid(alpha=0.3)
history.round(3)